# Vector Similarity Search (VSS)

In [8]:
import sqlite3
import numpy as np

In [9]:
# Create a connection to the database, create a db if it doesn't exists
conn = sqlite3.connect('vectors.db')

In [10]:
cursor = conn.cursor()

# Create the table with id and vector blob, blob is a binary data type that can store any binary data like serialized vectors
cursor.execute(
"""
CREATE TABLE IF NOT EXISTS vectors (
    id INTEGER PRIMARY KEY,
    vector BLOB NOT NULL
)
"""
)

In [11]:
# Create two sample vector
vect1 = np.array([1.2, 3.4, 2.1, 0.8])
vect2 = np.array([2.7, 1.5, 3.9, 2.3])

# Insert the vectors into the table
cursor.execute("INSERT INTO vectors (vector) VALUES (?)", (sqlite3.Binary(vect1.tobytes()),))
cursor.execute("INSERT INTO vectors (vector) VALUES (?)", (sqlite3.Binary(vect2.tobytes()),))

# Commit the transaction
conn.commit()

In [12]:
vect1.tobytes() # Converts the vector to a byte string

b'333333\xf3?333333\x0b@\xcd\xcc\xcc\xcc\xcc\xcc\x00@\x9a\x99\x99\x99\x99\x99\xe9?'

In [13]:
cursor.execute("SELECT vector FROM vectors")
rows = cursor.fetchall()

# Convert the byte string back to a numpy array
vector = np.frombuffer(rows[0][0], dtype=np.float64)
vector

array([1.2, 3.4, 2.1, 0.8])

In [14]:
vectors = []

for row in rows:
    vector = np.frombuffer(row[0], dtype=np.float64)
    vectors.append(vector)

vectors
    

[array([1.2, 3.4, 2.1, 0.8]), array([2.7, 1.5, 3.9, 2.3])]

## Querying the vector

In [32]:
query_vector = np.array([1.0, 3.2, 2.0, 0.5])

# Use abs function to calculate the distance between vectors (needs to fetch data ascending)
cursor.execute("""
                SELECT 
                    vector 
                FROM 
                    vectors 
                ORDER BY abs(vector - ?) 
                ASC
            """
        , (sqlite3.Binary(query_vector.tobytes()),))

In [33]:
res = cursor.fetchone()
res = np.frombuffer(res[0], dtype=np.float64)
res

array([2.7, 1.5, 3.9, 2.3])